In [27]:
from pathlib import Path

import pandas as pd

DATA_PATH = Path(
    "../data/processed/dataset_solar_2023_2024_v3.parquet"
)

df = pd.read_parquet(DATA_PATH)

print(f"Filas: {df.shape[0]:,}")
print(f"Columnas: {df.shape[1]}")
df.head()

Filas: 1,052,640
Columnas: 27


,ano,mes_sin,mes_cos,dia,hora_sin,hora_cos,minuto,fecha,ghi,dni,...,periodo_solar,temperatura,velocidad_viento,humedad_relativa,direccion_viento_sin,direccion_viento_cos,var_meteo_imp,codigo_ghi,codigo_dni,codigo_dhi
0,2023,0.0,1.0,1,0.0,1.0,0,2023-01-01 00:00:00,0.0,0.0,...,noche,10.0,1.050,77.0,0.884988,-0.465615,False,0,0,0
1,2023,0.0,1.0,1,0.0,1.0,1,2023-01-01 00:01:00,0.0,0.0,...,noche,10.0,1.333,77.0,-0.748956,0.662620,False,0,0,0
2,2023,0.0,1.0,1,0.0,1.0,2,2023-01-01 00:02:00,0.0,0.0,...,noche,10.0,1.100,77.0,-0.972031,-0.234854,False,0,0,0
3,2023,0.0,1.0,1,0.0,1.0,3,2023-01-01 00:03:00,0.0,0.0,...,noche,10.0,1.333,77.0,-0.937179,0.348850,False,0,0,0
4,2023,0.0,1.0,1,0.0,1.0,4,2023-01-01 00:04:00,0.0,0.0,...,noche,10.0,1.058,77.0,-0.443587,0.896231,False,0,0,0


In [28]:
data_dictionary = pd.DataFrame({
    "column_name": df.columns,
    "pandas_dtype": df.dtypes.astype(str).values,
    "null_count": df.isna().sum().values,
    "null_percentage": (
        df.isna().mean().mul(100).round(4).values
    ),
    "unique_values": df.nunique(dropna=False).values,
})

data_dictionary

,column_name,pandas_dtype,null_count,null_percentage,unique_values
0,ano,int64,0,0.0000,2
1,mes_sin,float64,0,0.0000,11
2,mes_cos,float64,0,0.0000,11
3,dia,int64,0,0.0000,31
4,hora_sin,float64,0,0.0000,21
5,hora_cos,float64,0,0.0000,22
6,minuto,int64,0,0.0000,60
7,fecha,datetime64[us],0,0.0000,1052640
8,ghi,float64,11868,1.1275,38879
9,dni,float64,11868,1.1275,34273


In [29]:
numeric_summary = df.select_dtypes(
    include="number"
).describe().T

numeric_summary

,count,mean,std,min,25%,50%,75%,max
ano,1052640.0,2.023501e+03,0.500000,2023.000000,2023.000000,2.024000e+03,2024.000000,2024.000000
mes_sin,1052640.0,-2.785087e-03,0.706860,-1.000000,-0.500000,0.000000e+00,0.866025,1.000000
mes_cos,1052640.0,-3.554140e-03,0.707340,-1.000000,-0.866025,-1.836970e-16,0.500000,1.000000
dia,1052640.0,1.573871e+01,8.803925,1.000000,8.000000,1.600000e+01,23.000000,31.000000
hora_sin,1052640.0,1.188018e-18,0.707107,-1.000000,-0.707107,6.123234e-17,0.707107,1.000000
hora_cos,1052640.0,-8.256725e-17,0.707107,-1.000000,-0.707107,-6.123234e-17,0.707107,1.000000
minuto,1052640.0,2.950000e+01,17.318111,0.000000,14.750000,2.950000e+01,44.250000,59.000000
ghi,1040772.0,1.782728e+02,285.576990,0.000000,0.000000,0.000000e+00,293.800000,2525.500000
dni,1040772.0,2.288883e+02,341.553291,0.000000,0.000000,0.000000e+00,537.917000,1083.250000
dhi,1040772.0,7.015291e+01,135.608487,0.000000,0.000000,0.000000e+00,80.636000,1153.000000


In [30]:
categorical_columns = df.select_dtypes(
    include=["object", "category", "bool"]
).columns.tolist()

categorical_columns

C:\Users\zacar\AppData\Local\Temp\ipykernel_12656\243135077.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_columns = df.select_dtypes(


['irr_null', 'periodo_solar', 'var_meteo_imp']

In [31]:
for column in [categorical_columns]:
    print(f"\nColumna: {column}")
    print(df[column].value_counts(dropna=False).head(20))


Columna: ['irr_null', 'periodo_solar', 'var_meteo_imp']
irr_null  periodo_solar  var_meteo_imp
False     dia            False            524263
          noche          False            515908
True      dia            False              7841
          noche          False              3931
False     dia            True                347
          noche          True                254
True      dia            True                 81
          noche          True                 15
Name: count, dtype: int64


In [32]:
metadata_columns = {
    "ano": {
        "description": "Año al que pertenece la observación.",
        "unit": "año",
        "postgresql_type": "SMALLINT",
        "nullable": False,
        "primary_key_candidate": False,
        "quality_rule": "Valor entero positivo. Rango esperado según los años disponibles en el proyecto."
    },
    "mes_sin": {
        "description": "Transformación seno cíclica del mes del año.",
        "unit": "adimensional",
        "postgresql_type": "DOUBLE PRECISION",
        "nullable": False,
        "primary_key_candidate": False,
        "quality_rule": "Valor comprendido entre -1 y 1."
    },
    "mes_cos": {
        "description": "Transformación coseno cíclica del mes del año.",
        "unit": "adimensional",
        "postgresql_type": "DOUBLE PRECISION",
        "nullable": False,
        "primary_key_candidate": False,
        "quality_rule": "Valor comprendido entre -1 y 1."
    },
    "dia": {
        "description": "Día del mes correspondiente a la observación.",
        "unit": "día",
        "postgresql_type": "SMALLINT",
        "nullable": False,
        "primary_key_candidate": False,
        "quality_rule": "Valor entero comprendido entre 1 y 31."
    },
    "hora_sin": {
        "description": "Transformación seno cíclica de la hora del día.",
        "unit": "adimensional",
        "postgresql_type": "DOUBLE PRECISION",
        "nullable": False,
        "primary_key_candidate": False,
        "quality_rule": "Valor comprendido entre -1 y 1."
    },
    "hora_cos": {
        "description": "Transformación coseno cíclica de la hora del día.",
        "unit": "adimensional",
        "postgresql_type": "DOUBLE PRECISION",
        "nullable": False,
        "primary_key_candidate": False,
        "quality_rule": "Valor comprendido entre -1 y 1."
    },
    "minuto": {
        "description": "Minuto dentro de la hora correspondiente a la observación.",
        "unit": "minuto",
        "postgresql_type": "SMALLINT",
        "nullable": False,
        "primary_key_candidate": False,
        "quality_rule": "Valor entero comprendido entre 0 y 59."
    },
    "fecha": {
        "description": "Fecha y hora exactas de la observación, con frecuencia temporal de un minuto.",
        "unit": "fecha y hora",
        "postgresql_type": "TIMESTAMP WITHOUT TIME ZONE",
        "nullable": False,
        "primary_key_candidate": True,
        "quality_rule": "Valor obligatorio y único para cada observación."
    },
    "ghi": {
        "description": "Irradiancia solar global medida sobre una superficie horizontal.",
        "unit": "W/m²",
        "postgresql_type": "DOUBLE PRECISION",
        "nullable": True,
        "primary_key_candidate": False,
        "quality_rule": "Puede ser nulo cuando la medición original no está disponible. La validación física se realiza en Python."
    },
    "dni": {
        "description": "Irradiancia solar directa normal medida perpendicularmente a los rayos solares.",
        "unit": "W/m²",
        "postgresql_type": "DOUBLE PRECISION",
        "nullable": True,
        "primary_key_candidate": False,
        "quality_rule": "Puede ser nulo cuando la medición original no está disponible. La validación física se realiza en Python."
    },
    "dhi": {
        "description": "Irradiancia solar difusa medida sobre una superficie horizontal.",
        "unit": "W/m²",
        "postgresql_type": "DOUBLE PRECISION",
        "nullable": True,
        "primary_key_candidate": False,
        "quality_rule": "Puede ser nulo cuando la medición original no está disponible. La validación física se realiza en Python."
    },
    "ghi_estimado": {
        "description": "Valor estimado de la irradiancia global horizontal obtenido a partir de las componentes directa, difusa y de la elevación solar.",
        "unit": "W/m²",
        "postgresql_type": "DOUBLE PRECISION",
        "nullable": True,
        "primary_key_candidate": False,
        "quality_rule": "Debe ser nulo cuando no existan datos suficientes para calcularlo."
    },
    "irr_null": {
        "description": "Indicador de existencia de valores nulos en las variables de irradiancia de la observación.",
        "unit": "booleano",
        "postgresql_type": "BOOLEAN",
        "nullable": False,
        "primary_key_candidate": False,
        "quality_rule": "Solo admite los valores TRUE o FALSE."
    },
    "error_balance": {
        "description": "Diferencia con signo entre la irradiancia global horizontal medida y la irradiancia global horizontal estimada.",
        "unit": "W/m²",
        "postgresql_type": "DOUBLE PRECISION",
        "nullable": True,
        "primary_key_candidate": False,
        "quality_rule": "Debe ser nulo cuando no pueda calcularse el balance de irradiancia."
    },
    "error_balance_abs": {
        "description": "Valor absoluto del error entre la irradiancia global horizontal medida y la estimada.",
        "unit": "W/m²",
        "postgresql_type": "DOUBLE PRECISION",
        "nullable": True,
        "primary_key_candidate": False,
        "quality_rule": "Valor mayor o igual que cero cuando no sea nulo."
    },
    "error_balance_rel": {
        "description": "Error relativo entre la irradiancia global horizontal medida y la estimada.",
        "unit": "adimensional",
        "postgresql_type": "DOUBLE PRECISION",
        "nullable": True,
        "primary_key_candidate": False,
        "quality_rule": "Debe ser nulo cuando no pueda calcularse el error relativo."
    },
    "elevacion_solar": {
        "description": "Ángulo de elevación del Sol sobre el horizonte en el instante de la observación.",
        "unit": "grados",
        "postgresql_type": "DOUBLE PRECISION",
        "nullable": False,
        "primary_key_candidate": False,
        "quality_rule": "Valor comprendido entre -90 y 90 grados."
    },
    "periodo_solar": {
        "description": "Clasificación del instante según exista o no presencia solar sobre el horizonte.",
        "unit": "categoría",
        "postgresql_type": "VARCHAR(10)",
        "nullable": False,
        "primary_key_candidate": False,
        "quality_rule": "Solo admite las categorías definidas en el dataset para periodo diurno y nocturno."
    },
    "temperatura": {
        "description": "Temperatura del aire asociada al instante de la observación.",
        "unit": "°C",
        "postgresql_type": "DOUBLE PRECISION",
        "nullable": True,
        "primary_key_candidate": False,
        "quality_rule": "Puede ser nulo cuando el dato meteorológico no está disponible o no ha podido imputarse."
    },
    "velocidad_viento": {
        "description": "Velocidad del viento asociada al instante de la observación.",
        "unit": "m/s",
        "postgresql_type": "DOUBLE PRECISION",
        "nullable": True,
        "primary_key_candidate": False,
        "quality_rule": "Valor mayor o igual que cero cuando no sea nulo."
    },
    "humedad_relativa": {
        "description": "Humedad relativa del aire asociada al instante de la observación.",
        "unit": "%",
        "postgresql_type": "DOUBLE PRECISION",
        "nullable": True,
        "primary_key_candidate": False,
        "quality_rule": "Valor comprendido entre 0 y 100 cuando no sea nulo."
    },
    "direccion_viento_sin": {
        "description": "Transformación seno cíclica de la dirección del viento.",
        "unit": "adimensional",
        "postgresql_type": "DOUBLE PRECISION",
        "nullable": True,
        "primary_key_candidate": False,
        "quality_rule": "Valor comprendido entre -1 y 1 cuando no sea nulo."
    },
    "direccion_viento_cos": {
        "description": "Transformación coseno cíclica de la dirección del viento.",
        "unit": "adimensional",
        "postgresql_type": "DOUBLE PRECISION",
        "nullable": True,
        "primary_key_candidate": False,
        "quality_rule": "Valor comprendido entre -1 y 1 cuando no sea nulo."
    },
    "var_meteo_imp": {
        "description": "Indicador de que una o varias variables meteorológicas de la observación han sido imputadas.",
        "unit": "booleano",
        "postgresql_type": "BOOLEAN",
        "nullable": False,
        "primary_key_candidate": False,
        "quality_rule": "Solo admite los valores TRUE o FALSE."
    },
    "codigo_ghi": {
        "description": (
            "Código de clasificación de la calidad de la medición de "
            "irradiancia global horizontal: 0 para dato correcto y "
            "1 para irregularidad o anomalía en la captación."
        ),
        "unit": "categoría codificada",
        "postgresql_type": "SMALLINT",
        "nullable": False,
        "primary_key_candidate": False,
        "quality_rule": "Solo admite los valores 0 y 1."
    },
    "codigo_dni": {
        "description": (
            "Código de clasificación de la calidad de la medición de "
            "irradiancia directa normal: 0 para dato correcto, "
            "1 para irregularidad o anomalía y 2 para condición de sombra "
            "asociada a una elevación solar igual o inferior a 5 grados."
        ),
        "unit": "categoría codificada",
        "postgresql_type": "SMALLINT",
        "nullable": False,
        "primary_key_candidate": False,
        "quality_rule": "Solo admite los valores 0, 1 y 2."
    },
    "codigo_dhi": {
        "description": (
            "Código de clasificación de la calidad de la medición de "
            "irradiancia difusa horizontal: 0 para dato correcto, "
            "1 para irregularidad o anomalía y 2 para condición de sombra "
            "asociada a una elevación solar igual o inferior a 5 grados."
        ),
        "unit": "categoría codificada",
        "postgresql_type": "SMALLINT",
        "nullable": False,
        "primary_key_candidate": False,
        "quality_rule": "Solo admite los valores 0, 1 y 2."
    }
}

In [33]:
metadata_df = (
    pd.DataFrame
    .from_dict(metadata_columns, orient="index")
    .reset_index()
    .rename(columns={"index": "column_name"})
)

data_dictionary = data_dictionary.merge(
    metadata_df,
    on="column_name",
    how="left"
)

In [34]:
data_dictionary = data_dictionary[
    [
        "column_name",
        "description",
        "unit",
        "pandas_dtype",
        "postgresql_type",
        "null_count",
        "null_percentage",
        "unique_values",
        "nullable",
        "primary_key_candidate",
        "quality_rule"
    ]
]

data_dictionary

,column_name,description,unit,pandas_dtype,postgresql_type,null_count,null_percentage,unique_values,nullable,primary_key_candidate,quality_rule
0,ano,Año al que pertenece la observación.,año,int64,SMALLINT,0,0.0000,2,False,False,Valor entero positivo. Rango esperado según lo...
1,mes_sin,Transformación seno cíclica del mes del año.,adimensional,float64,DOUBLE PRECISION,0,0.0000,11,False,False,Valor comprendido entre -1 y 1.
2,mes_cos,Transformación coseno cíclica del mes del año.,adimensional,float64,DOUBLE PRECISION,0,0.0000,11,False,False,Valor comprendido entre -1 y 1.
3,dia,Día del mes correspondiente a la observación.,día,int64,SMALLINT,0,0.0000,31,False,False,Valor entero comprendido entre 1 y 31.
4,hora_sin,Transformación seno cíclica de la hora del día.,adimensional,float64,DOUBLE PRECISION,0,0.0000,21,False,False,Valor comprendido entre -1 y 1.
5,hora_cos,Transformación coseno cíclica de la hora del día.,adimensional,float64,DOUBLE PRECISION,0,0.0000,22,False,False,Valor comprendido entre -1 y 1.
6,minuto,Minuto dentro de la hora correspondiente a la ...,minuto,int64,SMALLINT,0,0.0000,60,False,False,Valor entero comprendido entre 0 y 59.
7,fecha,"Fecha y hora exactas de la observación, con fr...",fecha y hora,datetime64[us],TIMESTAMP WITHOUT TIME ZONE,0,0.0000,1052640,False,True,Valor obligatorio y único para cada observación.
8,ghi,Irradiancia solar global medida sobre una supe...,W/m²,float64,DOUBLE PRECISION,11868,1.1275,38879,True,False,Puede ser nulo cuando la medición original no ...
9,dni,Irradiancia solar directa normal medida perpen...,W/m²,float64,DOUBLE PRECISION,11868,1.1275,34273,True,False,Puede ser nulo cuando la medición original no ...


In [35]:
OUTPUT_PATH = Path("../docs/data_dictionary_v3.csv")

OUTPUT_PATH.parent.mkdir(
    parents=True,
    exist_ok=True
)

data_dictionary.to_csv(
    OUTPUT_PATH,
    index=False,
    encoding="utf-8-sig"
)